In [27]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from graphviz import Digraph

In [28]:
class DecisionTreeID3:
    def __init__(self, max_depth=None):
        self.max_depth = max_depth
        self.tree = None

    def fit(self, X, y):
        features = list(range(X.shape[1]))
        self.tree = self._build_tree(X, y, features, depth=0)

    def _entropy(self, y):
        n = len(y)
        _, counts = np.unique(y, return_counts=True)
        probs = counts / n

        return -np.sum(probs * np.log2(probs + 1e-9))
    
    def _infomation_gain(self, X, y):
        parent_entropy = self._entropy(y)
        n = len(y)

        weighted = 0.0
        for val in np.unique(X):
            y_sub = y[X == val]
            weighted += (len(y_sub) / n) * self._entropy(y_sub)

        return parent_entropy - weighted
    
    def _best_feature(self, X, y, features):
        gains = [self._infomation_gain(X[:,f], y) for f in features]
        return features[int(np.argmax(gains))]
    
    def _majority_class(self, y):
        classes, counts = np.unique(y, return_counts=True)
        return classes[np.argmax(counts)]

    def _build_tree(self, X, y, features, depth):
        if len(np.unique(y)) == 1:
            return {'leaf': True, 'label': y[0]}

        if not features:                              
            return {'leaf': True, 'label': self._majority_class(y)}

        if self.max_depth is not None and depth >= self.max_depth:
            return {'leaf': True, 'label': self._majority_class(y)}
        
        best_f = self._best_feature(X, y, features)

        remaining = [f for f in features if f != best_f]

        branches = {}
        for val in np.unique(X[:, best_f]):
            mask = X[:, best_f] == val
            X_sub, y_sub = X[mask], y[mask]

            if len(y_sub) == 0:
                branches[val] = {'leaf': True, 'label': self._majority_class(y)}
            else:
                branches[val] = self._build_tree(X_sub, y_sub, remaining, depth + 1)
        return {
            'leaf': False,
            'feature': best_f,
            'branches': branches,
            'default': self._majority_class(y) 
        }
    
    def predict(self, X):
        return np.array([self._traverse(self.tree, x) for x in X])

    def _traverse(self, node, x):
        if node['leaf']: 
            return node['label']
        
        val = x[node['feature']]
        if val not in node['branches']:
            return node['default']

        return self._traverse(node['branches'][val], x)
    
    def visualize_tree(self):
        dot = Digraph()

        def add_nodes(node, parent=None, edge_label=""):
            node_id = str(id(node))

            if node['leaf']:
                dot.node(node_id, f"Leaf: {node['label']}")
            else:
                dot.node(node_id, f"Feature {node['feature']}")

            if parent:
                dot.edge(parent, node_id, label=edge_label)

            if not node['leaf']:
                for val, child in node['branches'].items():
                    add_nodes(child, node_id, str(val))

        add_nodes(self.tree)
        return dot

In [29]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import KBinsDiscretizer

X, y = load_iris(return_X_y=True)

kbd = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='uniform')
X_disc = kbd.fit_transform(X).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X_disc, y, random_state=42)

model = DecisionTreeID3(max_depth=5)
model.fit(X_train, y_train)
acc = (model.predict(X_test) == y_test).mean()
print(f"Accuracy: {acc:.2%}")   

dot = model.visualize_tree()
dot.render("tree", format="png")

Accuracy: 97.37%


'tree.png'